# 06 · Clean HPD violations
**Input:** `data/raw/hpd_violations/` · **Output:** `data/processed/hpd_violations_clean.parquet`

Steps:
1. Drop duplicate ids.
2. Parse dates (January 2025–October 2026 inspections).
3. Validate the BBL.
4. Keep classes A/B/C/I.
5. **Tag the free-text description with regex** into topics, then map the topics onto the same problem types as complaints.
6. Add flags: open, lead paint, rent-impairing.

**Study window: January 1, 2025–October 31, 2026.** Annual comparisons should use January–October in both years. Housing prediction uses six-month histories and four-month outcomes; older figures and reports describe the previous study.

In [1]:
import sys, pathlib, json
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config, housing as H, clean_housing as C
from src.preprocess import CleaningReport

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_colwidth", 70)

def show_report(rep):
    return pd.DataFrame(rep.steps)[["step", "rows_before", "rows_after", "rows_removed", "detail"]]

def save(df, name, rep):
    df.to_parquet(C.OUT[name], index=False)
    (config.REPORTS_DIR / f"cleaning_report_{name}.json").write_text(json.dumps(rep.to_dict(), indent=2, default=str))
    print(f"Saved {len(df):,} rows × {df.shape[1]} columns → {C.OUT[name].relative_to(ROOT)}")

In [2]:
raw = H.load("hpd_violations")
rep = CleaningReport("hpd_violations")
clean = C.clean_violations(raw, rep)
show_report(rep)

,step,rows_before,rows_after,rows_removed,detail
0,load_raw,1670801,1670801,0,
1,drop_duplicate_violationid,1670801,1670801,0,
2,parse_dates,1670801,1670801,0,missing or pre-2025 inspectiondate
3,valid_bbl,1670801,1669257,1544,missing or invalid BBL
4,valid_class,1669257,1669257,0,class not in A/B/C/I
5,derive_fields,1669257,1669257,0,"topic via regex, problem_type, open/lead/rent-impairing flags"


### Regex tagging: the patterns, and how much they cover

In [3]:
display(pd.Series(H.VIOLATION_TOPICS, name="regex pattern").to_frame())
print(f"Tagged with a topic: {(clean['topic'] != 'other').mean():.1%}")
pd.crosstab(clean["problem_type"], clean["class"], margins=True).sort_values("All", ascending=False)

,regex pattern
heat/hot water,\bHEAT|HOT WATER|BOILER
pests,ROACH|MICE|\bRATS?\b|RODENT|VERMIN|BED ?BUG|INSECT|PEST
mold/leaks,\bMOLD|MOULD|LEAK|WATER DAMAGE|DAMP
lead paint,LEAD
paint/plaster,PAINT|PLASTER
fire safety,SMOKE|CARBON MONOXIDE|FIRE|SPRINKLER|EGRESS
windows/guards,WINDOW
doors/locks,DOOR|LOCK
electric,ELECTRIC|OUTLET|WIRING|LIGHT
plumbing,PLUMB|TOILET|SINK|FAUCET|DRAIN|BASIN|BATH


Tagged with a topic: 91.2%


class,A,B,C,I,All
problem_type,,,,,
All,407826,688495,501463,71473,1669257
paint/plaster,112601,126416,64219,0,303236
pests,146526,11552,112965,0,271043
other,48268,97644,27867,71390,245169
doors/windows,13080,82741,145802,80,241703
leaks/mold,49684,118587,17355,0,185626
safety,15558,145117,22086,1,182762
heat/hot water,1718,4986,94728,1,101433
plumbing,14536,65745,9587,0,89868


In [4]:
# Spot-check: three random descriptions per problem type
for pt, g in clean.merge(raw[["violationid", "novdescription"]], on="violationid").groupby("problem_type"):
    print(f"\n[{pt}]")
    for t in g["novdescription"].dropna().sample(min(3, len(g)), random_state=0):
        print("   ", str(t)[:140])


[doors/windows]
    § 27-2005, 27-2007, 27-2041.1 HMC, §238, § 309; § 107 (2) ( C) MDL AND 28 RCNY §25-171: REPLACE OR REPAIR THE SELF-CLOSING DOORS THAT IS MIS
    § 27-2005 HMC: PROPERLY REPAIR OR REPLACE THE BROKEN OR DEFECTIVE SPRING BALANCE AT UPPER WINDOW SASH IN THE KITCHEN LOCATED AT APT 201, 2nd
    § 27-2005 HMC: PROPERLY REPAIR OR REPLACE THE BROKEN OR DEFECTIVE HEAVY DUTY MORTISE LOCK ASSEMBLY AT BUILDING ENTRANCE DOOR , 1st STORY

[electric]


    § 27-2005 HMC: PROPERLY REPAIR OR REPLACE THE BROKEN OR DEFECTIVE ELECTRICAL OUTLET AT EAST WALL IN THE 1st ROOM FROM NORTH LOCATED AT APT 3
    § 27-2037, 2038 HMC: PROVIDE A SAFE AND ADEQUATE SUPPLY OF ELECTRIC SERVICE TO THE FIXTURES .. IN THE ENTIRE APARTMENT LOCATED AT APT FIRSTF
    § 27-2037, 2038 HMC: PROVIDE A SAFE AND ADEQUATE SUPPLY OF ELECTRIC SERVICE TO THE FIXTURES ,, IN THE ENTIRE APARTMENT LOCATED AT APT BB, 1s

[heat/hot water]
    § 27-2033 ADM CODE PROVIDE READY ACCESS TO BUILDINGS HEATING SYSTEM DOOR LOCKED AT WEST AT BOILER ROOM
    § 27-2031 ADM CODE PROVIDE HOT WATER AT ALL HOT WATER FIXTURES IN THE ENTIRE APARTMENT LOCATED AT APT 14, 2nd STORY, 1st APARTMENT FROM WEST
    § 27-2031 ADM CODE PROVIDE HOT WATER AT ALL HOT WATER FIXTURES IN THE ENTIRE APARTMENT LOCATED AT APT 2B, 3rd STORY, 1st APARTMENT FROM WEST

[leaks/mold]
    § 27-2017.3 HMC: TRACE AND REPAIR THE SOURCE AND ABATE THE VISIBLE MOLD CONDITION... LESS THAN 10SQ FEET AT THE 1st WINDOW FRAME FRO

    § 27-2005 HMC: PROPERLY REPAIR OR REPLACE THE BROKEN OR DEFECTIVE WOOD BASEBOARD AT WEST AND EAST WALLS IN THE PRIVATE HALLWAY LOCATED AT AP
    § 27-2053 ADM CODE POST SIGN ON WALL OF ENTRANCE STORY BEARING NAME, ADDRESS INCLUDING APARTMENT NUMBER IF ANY, AND TELEPHONE NUMBER OF SUPE
    §27-2107 ADM CODE OWNER FAILED TO FILE A VALID REGISTRATION STATEMENT WITH THE DEPARTMENT AS REQUIRED BY ADM CODE §27-2097 AND IS THEREFORE 

[paint/plaster]
    § 27-2056.6 ADM CODE - CORRECT THE LEAD-BASED PAINT HAZARD-PAINT THAT IS XRF TESTED INCONCLUSIVE FOR LEAD CONTENT AT 0.5MG/CM2 AND IS PRESUM
    § 27-2056.6 ADM CODE - CORRECT THE LEAD-BASED PAINT HAZARD-PAINT THAT IS XRF TESTED INCONCLUSIVE FOR LEAD CONTENT AT 0.5MG/CM2 AND IS PRESUM
    § 27-2005 ADM CODE REPAIR THE BROKEN OR DEFECTIVE PLASTERED SURFACES AND PAINT IN A UNIFORM COLOR WEST WALL IN THE KITCHEN LOCATED AT APT 1C

[pests]
    (A) § HMC:FILE ANNUAL BEDBUG REPORT IN ACCORDANCE WITH HPD RULE AS DESCRIBED ON THE BACK OF THIS NOT

    § 27-2026 ADM CODE REPAIR THE BROKEN OR DEFECTIVE CONNECTION WASH BASIN IN THE BATHROOM LOCATED AT APT 8M, 7th STORY, 6th APARTMENT FROM WES
    § 27-2005 ADM CODE PROPERLY REPAIR WITH SIMILAR MATERIAL THE BROKEN OR DEFECTIVE CERAMIC WALL TILES AT WEST IN THE BATHROOM LOCATED AT APT C
    § 27-2005 ADM CODE PROPERLY REPAIR WITH SIMILAR MATERIAL THE BROKEN OR DEFECTIVE CERAMIC WALL TILES AT THE NORTH WALL AND THE SOUTH WALL IN 

[safety]
    § 27-2045(B)(1)(B) HMC, § 12-06, § 12-07, § 12-09 RCNY REPAIR, REPLACE OR PROVIDE AN APPROVED AND OPERATIONAL CARBON MONOXIDE DETECTING DEVI
    § 27-2005 ADM CODE & 309 M/D LAW ABATE THE NUISANCE CONSISTING OF AC UNIT INSTALLED AT FIRE ESCAPE WINDOW IN THE 4th ROOM FROM EAST LOCATED 
    § 27-2005 ADM CODE & 309 M/D LAW ABATE THE NUISANCE CONSISTING OF SIDEWALK SCAFFOLD SHED BLOCKING ACCESS TO FIRE ESCAPE WINDOW IN THE 1st RO


In [5]:
print(f"Open: {clean['is_open'].mean():.1%} | class C: {(clean['class'] == 'C').mean():.1%} | "
      f"lead: {clean['is_lead'].mean():.1%} | rent-impairing: {clean['is_rent_impairing'].mean():.1%}")
save(clean, "hpd_violations", rep)

Open: 45.3% | class C: 30.0% | lead: 3.7% | rent-impairing: 7.6%


Saved 1,669,257 rows × 14 columns → data\processed\hpd_violations_clean.parquet
